### Setup

In [ ]:
import ast
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

df = pd.read_csv('/Users/netaspektor/School/data-mining/songs/kaggle.csv')
df = df.rename(columns={'songs': 'lyrics'})

### Clean

In [ ]:
# remove where lyrics are null
df = df.dropna(subset=['lyrics'])
df = df.loc[df["lyrics"] != "[]"]

In [ ]:
df.head()

In [ ]:
# change lyrics to actual list
df["lyrics_list"] = df["lyrics"].apply(ast.literal_eval)

In [ ]:
total_init_obs = len(df)
initial_song_count = df['lyrics'].nunique()
initial_singer_count = df['artist_key'].nunique()
print(f"excluding na's, there are initialy {total_init_obs} observations in the dataset")
print(f"there are initialy {initial_song_count} songs in the dataset")
print(f"there are initialy {initial_singer_count} singers in the dataset")

We noticed a weird observation where all words in the lyrics are two letters (probably only parts of the original lyrics). So we wanted to check the histogram of avg length of words in songs, expecting it to be distributed normally.

In [ ]:
df["avg_word_len"] = df["lyrics_list"].apply(
    lambda words: np.mean([len(w) for w in words]) if len(words) > 0 else np.nan
)
plt.figure(figsize=(8, 5))
plt.hist(df["avg_word_len"].dropna(), bins=40)
plt.xlabel("Average word length in song")
plt.ylabel("Number of songs")
plt.title("Distribution of average word length per song")
plt.show()

It seems there is a suspicious amount of songs where the average length is 2,  but the rest seems to distribute as expected.

In [ ]:
# remove songs where more than 70% of words are 2 letters (scraper split words into fragments)
def mostly_two_letter_words(words):
    if len(words) < 2:
        return False
    return sum(len(w) == 2 for w in words) / len(words) > 0.70

garbled_mask = df['lyrics_list'].apply(mostly_two_letter_words)
print("mostly-2-letter-word rows removed:", int(garbled_mask.sum()))
df = df[~garbled_mask]

In [ ]:
# data-quality scan: the URL's prfid identifies the performer's page on shironet, so a
# single prfid mapped to >1 artist_key means the same page was labelled as two artists.
# this is what flags the cases handled below (prfid=383 and prfid=666).
df['prfid'] = df['url'].str.extract(r'prfid=(\d+)')
prfid_key_counts = df.groupby('prfid')['artist_key'].nunique()
flagged = prfid_key_counts[prfid_key_counts > 1]
print("prfids attributed to >1 artist_key:", len(flagged))
for p in flagged.index:
    keys = df.loc[df['prfid'] == p, 'artist_key'].value_counts()
    print(f"  prfid={p}: " + ", ".join(f"{k}={n}" for k, n in keys.items()))

In [ ]:
# prfid=383 is Hava Alberstien but for some reason there are dupes linked to Ofer Levi, so we drop them
HAVA_KEY = 'Artist_Hava_Alberstien'
mislabeled = df['url'].str.contains('prfid=383', na=False) & (df['artist_key'] != HAVA_KEY)
df = df[~mislabeled]

In [ ]:
# prfid=666 is Margol, but she has dupes under two names: Margalit Tzanani and Margol.
# So we merge the lables and then dedupe.
df.loc[df['artist_key'] == 'Artist_Margol', 'artist_key'] = 'Artist_Margalit_Tzanani'
df = df.drop_duplicates(subset=['artist_key', 'url'])

In [ ]:
# dupes: check for same lyrics, different title, same artist
dupes = (
    df.groupby(['artist_key', 'lyrics'])
      .filter(lambda x: x['song'].nunique() > 1)
)
dupes_unique_lyrics = dupes['lyrics'].nunique()
print("case 1: unique lyrics count:", dupes_unique_lyrics)
print("case 1: effeced rows:", len(dupes))

In [ ]:
# drop the redundant copies
# among the case-1 rows, mark every row after the first of each lyric for removal
dupes_rows_to_drop = dupes.index.difference(
    dupes.drop_duplicates(subset=['lyrics']).index
)
df = df.drop(index=dupes_rows_to_drop)

In [ ]:
total_obs = len(df)
song_count = df['lyrics'].nunique()
singer_count = df['artist_key'].nunique()
print(f"after cleaning, there are {total_obs} observations in the dataset, {total_init_obs-total_obs} removed")
print(f"after cleaning, there are {song_count} songs in the dataset, {initial_song_count - song_count} removed")
print(f"after cleaning, there are {singer_count} singers in the dataset, {initial_singer_count - singer_count} removed")

In [202]:
# save cleaned df
df.to_csv('clean_df.csv', index=False, encoding='utf-8-sig')

In [ ]:
# ---- export the song list to enrich with Spotify (feeds spotify_enrich.py) ----
# keyed only on artist_key -- the script romanizes it ("Artist_Adam" -> "Adam") for the query.
clean = df[['artist_key', 'song', 'lyrics']]
clean.to_csv('clean.csv', index=False, encoding='utf-8-sig')